# Sandbox: Adjudicacion de Evidencia

Notebook interactivo para probar el flujo completo del contrato `DevContractShield`.
Modifica la evidencia en la **Celda 5** y re-ejecuta para ver como responde el adjudicador.

**Prerequisitos:**
- `GENLAYER_PRIVATE_KEY` con fondos en testnet Bradbury
- Faucet: https://faucet-bradbury.genlayer.com

## 1. Setup y conexion

In [1]:
import json
import os
from pathlib import Path

from genlayer_py import create_account, create_client
from genlayer_py.chains import studionet
from genlayer_py.types.transactions import TransactionStatus

private_key = os.environ.get("GENLAYER_PRIVATE_KEY")
if not private_key:
    raise EnvironmentError("Falta GENLAYER_PRIVATE_KEY. Obtener de https://faucet-bradbury.genlayer.com")

account = create_account(private_key)
client = create_client(chain=studionet, account=account)

CONTRACT_PATH = Path("dev_contract_shield.py")
assert CONTRACT_PATH.exists(), f"No se encuentra {CONTRACT_PATH}"

print(f"Private Key: {private_key}")
print(f"Cuenta: {account.address}")
print(f"Contrato: {CONTRACT_PATH}")
print("Conectado a studionet")

Private Key: 0x7f45a8658b75af9095651c1916d94dcb3e634d54101ec474c4179cd0ed3267ff
Cuenta: 0x9452fffFC741787a56718B75BfB76245dd2714C3
Contrato: dev_contract_shield.py
Conectado a studionet


## 2. Deploy del contrato

Cada ejecucion despliega una instancia nueva. Modifica los parametros si quieres.

In [2]:
# --- Parametros del contrato (editar si quieres) ---
FAKE_CLIENT    = "0x1111111111111111111111111111111111111111"
FAKE_DEVELOPER = "0x2222222222222222222222222222222222222222"
ESCROW_AMOUNT  = "1000"
STACK          = "react-javascript"
TITLE          = "Landing Page E-commerce"
DESCRIPTION    = "Desarrollo de landing page responsive con carrito de compras"
ACCEPTANCE_RULE = (
    "ACCEPTED si layer_a_pass_rate == 1.0 AND layer_c_status == 'pass'. "
    "REJECTED en cualquier otro caso. layer_b es informativo y NO afecta el resultado."
)

# --- Deploy ---
contract_code = CONTRACT_PATH.read_bytes()

print("Desplegando contrato...")
tx_hash = client.deploy_contract(
    code=contract_code,
    args=[FAKE_CLIENT, FAKE_DEVELOPER, ESCROW_AMOUNT, STACK, TITLE, DESCRIPTION, ACCEPTANCE_RULE],
)
print(f"Deploy tx: {tx_hash}")

receipt = client.wait_for_transaction_receipt(
    transaction_hash=tx_hash,
    status=TransactionStatus.FINALIZED,
    retries=30,
    interval=5000,
)

# Extraer address
if isinstance(receipt, dict):
    data = receipt.get("data", receipt)
else:
    data = getattr(receipt, "data", receipt)

contract_address = data.get("contract_address", "") if isinstance(data, dict) else getattr(data, "contract_address", "")

if not contract_address:
    raise Exception(f"No se encontro contract_address en receipt: {receipt}")

print(f"Contract address: {contract_address}")

# Verificar estado inicial
state = client.read_contract(address=contract_address, function_name="get_state", args=[])
print(f"Estado: {state}")

Desplegando contrato...
Deploy tx: 0x27944436dcbe1f07896edc25c2e5cdca99ccf50bc4d7fc87cc98d0dab69ff2d0
Contract address: 0x402B911B10c26756051D382626053C253c0a2F44
Estado: Created


## 3. Fondear contrato

In [3]:
FUND_AMOUNT = 1  # GEN a enviar

print(f"Fondeando con {FUND_AMOUNT} GEN...")
tx_hash = client.write_contract(
    address=contract_address,
    function_name="fund_contract",
    args=[],
    value=FUND_AMOUNT,
)
client.wait_for_transaction_receipt(
    transaction_hash=tx_hash,
    status=TransactionStatus.FINALIZED,
    retries=30,
    interval=5000,
)

state = client.read_contract(address=contract_address, function_name="get_state", args=[])
print(f"Estado: {state}")

Fondeando con 1 GEN...
Estado: Funded


## 4. Registrar hashes

In [4]:
# Test pack hash
print("Registrando test pack hash...")
tx_hash = client.write_contract(
    address=contract_address,
    function_name="set_test_pack_hash",
    args=["sha256:def456"],
)
client.wait_for_transaction_receipt(
    transaction_hash=tx_hash,
    status=TransactionStatus.FINALIZED,
    retries=30,
    interval=5000,
)
print("Test pack hash registrado")

# Submission hash (cambia estado a Submitted)
print("Registrando submission hash...")
tx_hash = client.write_contract(
    address=contract_address,
    function_name="set_submission_hash",
    args=["sha256:abc123"],
)
client.wait_for_transaction_receipt(
    transaction_hash=tx_hash,
    status=TransactionStatus.FINALIZED,
    retries=30,
    interval=5000,
)

state = client.read_contract(address=contract_address, function_name="get_state", args=[])
print(f"Estado: {state}")

Registrando test pack hash...
Test pack hash registrado
Registrando submission hash...
Estado: Submitted


## 5. Evidencia editable

**Esta es la celda clave.** Modifica los valores y re-ejecuta celdas 5 + 6 para probar distintos escenarios.

Regla binaria:
- `ACCEPTED` = `layer_a_pass_rate == 1.0` AND `layer_c_status == "pass"`
- `REJECTED` = cualquier otro caso
- `layer_b` es **informativo**, NO afecta el resultado

In [5]:
# ============================================================
#  EDITA AQUI para probar distintos escenarios
# ============================================================

evidence = {
    # --- Layer A: Tests del cliente (DEBE ser 1.0 para aceptar) ---
    "layer_a_pass_rate": 0.9,       # Cambiar a 0.75, 0.5, 0.0, etc.

    # --- Layer B: Tests informativos (NO afecta resultado) ---
    "layer_b_pass_rate": 0.85,
    "layer_b_informative_only": True,

    # --- Layer C: Linter + runner (DEBE ser "pass" para aceptar) ---
    "layer_c_status": "pass",       # Cambiar a "fail" para rechazar

    # --- Metadata ---
    "contract_id": "sandbox-001",
    "submission_id": "sub-sandbox-001",
    "language_or_stack": "react-javascript",
    "code_hash": "sha256:sandbox123",
    "test_pack_hash": "sha256:def456",
    "evidence_bundle_hash": "sha256:bundle_sandbox",
    "runner": {"name": "vitest", "version": "3.1.1"},
    "linter": {"name": "eslint", "version": "9.0.0"},
    "runtime": {
        "node": "23.11.1",
        "package_manager": "npm",
        "pm_version": "10.9.2",
        "lockfile_hash": "sha256:lock123",
    },
    "execution": {"command": "npm test", "timeout_seconds": 120},
    "proposed_result": "accepted",
}

evidence_json = json.dumps(evidence, indent=2)
print("Evidencia a enviar:")
print(evidence_json)
print(f"\n-> layer_a_pass_rate: {evidence['layer_a_pass_rate']}")
print(f"-> layer_c_status: {evidence['layer_c_status']}")
print(f"-> Resultado esperado: {'ACCEPTED' if evidence['layer_a_pass_rate'] == 1.0 and evidence['layer_c_status'] == 'pass' else 'REJECTED'}")

Evidencia a enviar:
{
  "layer_a_pass_rate": 0.9,
  "layer_b_pass_rate": 0.85,
  "layer_b_informative_only": true,
  "layer_c_status": "pass",
  "contract_id": "sandbox-001",
  "submission_id": "sub-sandbox-001",
  "language_or_stack": "react-javascript",
  "code_hash": "sha256:sandbox123",
  "test_pack_hash": "sha256:def456",
  "evidence_bundle_hash": "sha256:bundle_sandbox",
  "runner": {
    "name": "vitest",
    "version": "3.1.1"
  },
  "linter": {
    "name": "eslint",
    "version": "9.0.0"
  },
  "runtime": {
    "node": "23.11.1",
    "package_manager": "npm",
    "pm_version": "10.9.2",
    "lockfile_hash": "sha256:lock123"
  },
  "execution": {
    "command": "npm test",
    "timeout_seconds": 120
  },
  "proposed_result": "accepted"
}

-> layer_a_pass_rate: 0.9
-> layer_c_status: pass
-> Resultado esperado: REJECTED


## 6. Submit evidence

Envia la evidencia al contrato. El LLM on-chain adjudica.

In [6]:
print("Enviando evidencia al contrato...")
tx_hash = client.write_contract(
    address=contract_address,
    function_name="submit_evidence",
    args=[json.dumps(evidence)],
)
print(f"Evidence tx: {tx_hash}")

print("Esperando finalizacion (esto puede tardar ~30-60s)...")
evidence_receipt = client.wait_for_transaction_receipt(
    transaction_hash=tx_hash,
    status=TransactionStatus.FINALIZED,
    retries=30,
    interval=5000,
)

# Debug: mostrar info del receipt
if isinstance(evidence_receipt, dict):
    consensus = evidence_receipt.get("consensus_data", {})
    if isinstance(consensus, dict):
        leader = consensus.get("leader_receipt", [{}])
        if leader:
            first = leader[0] if isinstance(leader, list) else leader
            if isinstance(first, dict):
                print(f"Leader execution: {first.get('execution_result', 'N/A')}")
                genvm = first.get("genvm_result", {})
                if isinstance(genvm, dict):
                    if genvm.get("stderr"):
                        print(f"Leader stderr: {genvm['stderr'][:500]}")

print("Evidencia procesada.")

Enviando evidencia al contrato...
Evidence tx: 0xc03862a99acfac9f6ec8b85d1c554b597a30d9f64a67ace4a1b0defdc0b326dd
Esperando finalizacion (esto puede tardar ~30-60s)...
Leader execution: SUCCESS
Evidencia procesada.


In [8]:
tx = client.get_transaction("0xc803a2bf2adab937065bb3757ecf6822c2ecf5761350665f88942776bed75a02")                           
print(json.dumps(tx, indent=2, default=str)) 

{
  "hash": "0xc803a2bf2adab937065bb3757ecf6822c2ecf5761350665f88942776bed75a02",
  "from_address": "0x9452fffFC741787a56718B75BfB76245dd2714C3",
  "to_address": "0xbEE7fb0AF8d8e2E0668962339EEAFAcFaFf99fEe",
  "data": {
    "calldata": {
      "base64": "FgRhcmdzDcQoeyJsYXllcl9hX3Bhc3NfcmF0ZSI6IDAuOSwgImxheWVyX2JfcGFzc19yYXRlIjogMC44NSwgImxheWVyX2JfaW5mb3JtYXRpdmVfb25seSI6IHRydWUsICJsYXllcl9jX3N0YXR1cyI6ICJwYXNzIiwgImNvbnRyYWN0X2lkIjogInNhbmRib3gtMDAxIiwgInN1Ym1pc3Npb25faWQiOiAic3ViLXNhbmRib3gtMDAxIiwgImxhbmd1YWdlX29yX3N0YWNrIjogInJlYWN0LWphdmFzY3JpcHQiLCAiY29kZV9oYXNoIjogInNoYTI1NjpzYW5kYm94MTIzIiwgInRlc3RfcGFja19oYXNoIjogInNoYTI1NjpkZWY0NTYiLCAiZXZpZGVuY2VfYnVuZGxlX2hhc2giOiAic2hhMjU2OmJ1bmRsZV9zYW5kYm94IiwgInJ1bm5lciI6IHsibmFtZSI6ICJ2aXRlc3QiLCAidmVyc2lvbiI6ICIzLjEuMSJ9LCAibGludGVyIjogeyJuYW1lIjogImVzbGludCIsICJ2ZXJzaW9uIjogIjkuMC4wIn0sICJydW50aW1lIjogeyJub2RlIjogIjIzLjExLjEiLCAicGFja2FnZV9tYW5hZ2VyIjogIm5wbSIsICJwbV92ZXJzaW9uIjogIjEwLjkuMiIsICJsb2NrZmlsZV9oYXNoIjogInNoYTI1Njpsb2NrM

## 7. Resultado de adjudicacion

In [7]:
result = client.read_contract(
    address=contract_address,
    function_name="get_result",
    args=[],
)

info = client.read_contract(
    address=contract_address,
    function_name="get_contract_info",
    args=[],
)

print("=" * 50)
print("  RESULTADO DE ADJUDICACION")
print("=" * 50)
print(f"  Veredicto:  {result['result'].upper()}")
print(f"  Razon:      {result['reason']}")
print(f"  Estado:     {result['state']}")
print(f"  Propuesto:  {result.get('proposed_result', 'N/A')}")
print("=" * 50)

# Verificar contra expectativa
expected = "accepted" if evidence["layer_a_pass_rate"] == 1.0 and evidence["layer_c_status"] == "pass" else "rejected"
match = result["result"] == expected
print(f"\n  Esperado: {expected.upper()}")
print(f"  Match:    {'SI' if match else 'NO - DISCREPANCIA!'}")

if not match:
    print(f"\n  ATENCION: El LLM devolvio '{result['result']}' pero se esperaba '{expected}'")
    print(f"  Razon del LLM: {result['reason']}")

  RESULTADO DE ADJUDICACION
  Veredicto:  REJECTED
  Razon:      layer_a_pass_rate is 0.9 which is not 1.0, even though layer_c_status is 'pass'.
  Estado:     Rejected
  Propuesto:  accepted

  Esperado: REJECTED
  Match:    SI


## 8. Re-deploy rapido (helper)

Ejecuta esta celda para desplegar un contrato nuevo, fondearlo, y registrar hashes en un solo paso.
Luego vuelve a la **Celda 5** para editar la evidencia y la **Celda 6** para enviarla.

In [ ]:
def quick_deploy():
    """Deploy + fund + set hashes en un solo paso."""
    global contract_address

    print("[1/4] Desplegando contrato...")
    code = CONTRACT_PATH.read_bytes()
    tx = client.deploy_contract(
        code=code,
        args=[FAKE_CLIENT, FAKE_DEVELOPER, ESCROW_AMOUNT, STACK, TITLE, DESCRIPTION, ACCEPTANCE_RULE],
    )
    r = client.wait_for_transaction_receipt(transaction_hash=tx, status=TransactionStatus.FINALIZED, retries=30, interval=5000)
    data = r.get("data", r) if isinstance(r, dict) else getattr(r, "data", r)
    contract_address = data.get("contract_address", "") if isinstance(data, dict) else getattr(data, "contract_address", "")
    print(f"    Address: {contract_address}")

    print("[2/4] Fondeando...")
    tx = client.write_contract(address=contract_address, function_name="fund_contract", args=[], value=1)
    client.wait_for_transaction_receipt(transaction_hash=tx, status=TransactionStatus.FINALIZED, retries=30, interval=5000)

    print("[3/4] Test pack hash...")
    tx = client.write_contract(address=contract_address, function_name="set_test_pack_hash", args=["sha256:def456"])
    client.wait_for_transaction_receipt(transaction_hash=tx, status=TransactionStatus.FINALIZED, retries=30, interval=5000)

    print("[4/4] Submission hash...")
    tx = client.write_contract(address=contract_address, function_name="set_submission_hash", args=["sha256:abc123"])
    client.wait_for_transaction_receipt(transaction_hash=tx, status=TransactionStatus.FINALIZED, retries=30, interval=5000)

    state = client.read_contract(address=contract_address, function_name="get_state", args=[])
    print(f"\nListo! Address: {contract_address}")
    print(f"Estado: {state}")
    print("\n-> Ahora edita la Celda 5 y ejecuta Celda 6")

quick_deploy()